# Lab 2: 工具与执行层 — Tool Execution

对应 Harness 六层架构：第 ❷ 层

## 学习目标

- 理解 **统一工具接口** 设计模式（Factory Pattern）
- 实现 validation → execution → result 的工具管道
- 让 agent 能够真正读写文件、执行命令

## 痛点回顾

在 Lab 1 中，我们的 agent 有了 `tool_use` 意图，但无法执行——模型想调用工具，却只能收到错误。

**本 Lab 的目标：给 agent 装上真正的"手脚"。**

## 对应 Claude Code 源码

- `Tool.ts` — 统一工具接口定义（约 29KB）
- `tools/BashTool/` — Bash 命令执行工具
- `tools/FileReadTool/` — 文件读取工具
- `tools/FileWriteTool/` — 文件写入工具

Claude Code 有 30+ 个工具，但它们**全部实现同一个 Tool 接口**。我们将用 Python 复刻这个设计。


---
## 第一步：环境准备

In [ ]:
from anthropic import AnthropicBedrock
import subprocess
import os
from abc import ABC, abstractmethod

client = AnthropicBedrock(aws_region="us-west-2")
MODEL = "global.anthropic.claude-sonnet-4-6"

print(f"✓ 环境准备完成，模型: {MODEL}")

---
## 第二步：定义统一工具接口（Tool 基类）

Claude Code 中所有工具都实现同一个接口（`Tool.ts`）。核心方法：

| 方法 | 作用 |
|------|------|
| `name` | 工具名称，API 用来匹配 |
| `description` | 工具描述，让模型理解何时使用 |
| `input_schema` | JSON Schema，定义输入参数 |
| `validate()` | 校验输入是否合法 |
| `execute()` | 真正执行工具逻辑 |

这样的设计好处是：**新增工具只需实现这个接口，不需要改动 agent loop 代码。**

In [ ]:
# 统一工具接口（对应 Claude Code 的 Tool.ts）

class Tool(ABC):
    """所有工具的基类。对应 Claude Code 中 Tool.ts 的统一接口。"""
    
    @property
    @abstractmethod
    def name(self) -> str:
        """工具名称"""
        pass
    
    @property
    @abstractmethod
    def description(self) -> str:
        """工具描述（给模型看的）"""
        pass
    
    @property
    @abstractmethod
    def input_schema(self) -> dict:
        """输入参数的 JSON Schema"""
        pass
    
    def validate(self, tool_input: dict) -> bool:
        """校验输入是否合法（默认检查 required 字段）"""
        required = self.input_schema.get("required", [])
        return all(key in tool_input for key in required)
    
    @abstractmethod
    def execute(self, tool_input: dict) -> str:
        """执行工具，返回结果字符串"""
        pass
    
    def to_api_schema(self) -> dict:
        """转换为 Anthropic API 需要的 tool 格式"""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": self.input_schema,
        }

print("✓ Tool 基类定义完成")

---
## 第三步：实现三个核心工具

我们实现 Claude Code 中最核心的三个工具：

| 工具 | 功能 | 危险等级 |
|------|------|----------|
| `ReadFileTool` | 读取文件内容 | 低（只读） |
| `WriteFileTool` | 写入文件 | 中（有副作用） |
| `BashTool` | 执行 shell 命令 | 高（可做任何事） |

注意危险等级——这在 Lab 4（验证与安全层）中会变得很重要。


In [ ]:
# 工具 1：读取文件（对应 Claude Code 的 FileReadTool）

class ReadFileTool(Tool):
    @property
    def name(self):
        return "read_file"
    
    @property
    def description(self):
        return "读取指定路径的文件内容"
    
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "要读取的文件路径"}
            },
            "required": ["path"],
        }
    
    def execute(self, tool_input: dict) -> str:
        path = tool_input["path"]
        try:
            with open(path, "r") as f:
                content = f.read()
            return content if content else "(空文件)"
        except FileNotFoundError:
            return f"错误：文件 {path} 不存在"
        except Exception as e:
            return f"错误：{e}"

print("✓ ReadFileTool 实现完成")

In [ ]:
# 工具 2：写入文件（对应 Claude Code 的 FileWriteTool）

class WriteFileTool(Tool):
    @property
    def name(self):
        return "write_file"
    
    @property
    def description(self):
        return "将内容写入指定路径的文件。如果文件已存在会被覆盖。"
    
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "要写入的文件路径"},
                "content": {"type": "string", "description": "要写入的内容"},
            },
            "required": ["path", "content"],
        }
    
    def execute(self, tool_input: dict) -> str:
        path = tool_input["path"]
        content = tool_input["content"]
        try:
            # 自动创建目录
            os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
            with open(path, "w") as f:
                f.write(content)
            return f"已写入 {len(content)} 个字符到 {path}"
        except Exception as e:
            return f"错误：{e}"

print("✓ WriteFileTool 实现完成")

In [ ]:
# 工具 3：执行 Bash 命令（对应 Claude Code 的 BashTool）

class BashTool(Tool):
    @property
    def name(self):
        return "bash"
    
    @property
    def description(self):
        return "执行一条 bash 命令并返回标准输出和标准错误"
    
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "要执行的 bash 命令"}
            },
            "required": ["command"],
        }
    
    def execute(self, tool_input: dict) -> str:
        command = tool_input["command"]
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=30,
                cwd=os.getcwd(),
            )
            output = result.stdout + result.stderr
            return output if output.strip() else "(命令执行完毕，无输出)"
        except subprocess.TimeoutExpired:
            return "错误：命令执行超时（30秒）"
        except Exception as e:
            return f"错误：{e}"

print("✓ BashTool 实现完成")
print("\n⚠️ 注意：BashTool 没有任何安全检查！它可以执行任何命令，包括 rm -rf /")
print("   这个问题我们在 Lab 4 中解决。")


---
## 第四步：构建完整的 Agent Loop

现在我们把三个工具接入 Agent Loop，实现完整的 observe-think-act 循环：

```
用户输入 → Claude API (带 tools) → 收到 tool_use
    → tool_map 查找工具 → validate → execute → 收集 tool_result
    → 送回 API → 继续循环直到 stop_reason != tool_use
```

核心改动：当收到 `tool_use` 时，我们**不再返回错误，而是真正执行工具**。


In [ ]:
# 注册工具
TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]

# 工具查找表：name → Tool 实例
tool_map = {t.name: t for t in TOOLS}

# 转换为 API schema
tools_schema = [t.to_api_schema() for t in TOOLS]

print(f"✓ 已注册 {len(TOOLS)} 个工具: {list(tool_map.keys())}")

In [ ]:
# === 将 Tool + 工具保存到 utils/ ===
# 后续 Lab 可以直接: from utils.tools import Tool, ReadFileTool, WriteFileTool, BashTool

import os
os.makedirs('utils', exist_ok=True)

# 将工具类的完整实现写入 utils/tools.py
# （与上面 cell 中定义的代码完全一致）
tools_source = '''import subprocess
import os
from abc import ABC, abstractmethod


class Tool(ABC):
    \"\"\"所有工具的基类。对应 Claude Code 中 Tool.ts 的统一接口。\"\"\"
    @property
    @abstractmethod
    def name(self) -> str: pass
    @property
    @abstractmethod
    def description(self) -> str: pass
    @property
    @abstractmethod
    def input_schema(self) -> dict: pass
    def validate(self, tool_input: dict) -> bool:
        return all(k in tool_input for k in self.input_schema.get("required", []))
    @abstractmethod
    def execute(self, tool_input: dict) -> str: pass
    def to_api_schema(self) -> dict:
        return {"name": self.name, "description": self.description, "input_schema": self.input_schema}


class ReadFileTool(Tool):
    @property
    def name(self): return "read_file"
    @property
    def description(self): return "读取指定路径的文件内容"
    @property
    def input_schema(self):
        return {"type": "object", "properties": {"path": {"type": "string", "description": "文件路径"}}, "required": ["path"]}
    def execute(self, tool_input):
        try:
            with open(tool_input["path"], "r") as f: return f.read() or "(空文件)"
        except Exception as e: return f"错误：{e}"


class WriteFileTool(Tool):
    @property
    def name(self): return "write_file"
    @property
    def description(self): return "将内容写入指定路径的文件。如果文件已存在会被覆盖。"
    @property
    def input_schema(self):
        return {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}
    def execute(self, tool_input):
        try:
            os.makedirs(os.path.dirname(tool_input["path"]) or ".", exist_ok=True)
            with open(tool_input["path"], "w") as f: f.write(tool_input["content"])
            return f"已写入 {len(tool_input['content'])} 字符到 {tool_input['path']}"
        except Exception as e: return f"错误：{e}"


class BashTool(Tool):
    @property
    def name(self): return "bash"
    @property
    def description(self): return "执行一条 bash 命令并返回标准输出和标准错误"
    @property
    def input_schema(self):
        return {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}
    def execute(self, tool_input):
        try:
            r = subprocess.run(tool_input["command"], shell=True, capture_output=True, text=True, timeout=30, cwd=os.getcwd())
            output = r.stdout + r.stderr
            return output.strip() if output.strip() else "(无输出)"
        except subprocess.TimeoutExpired: return "错误：命令执行超时（30秒）"
        except Exception as e: return f"错误：{e}"
'''

with open('utils/tools.py', 'w') as f:
    f.write(tools_source)

# 验证
from importlib import reload
import utils.tools
reload(utils.tools)
from utils.tools import Tool as _T, ReadFileTool as _R, WriteFileTool as _W, BashTool as _B
print('✓ 已保存并验证 utils/tools.py')
print('  包含: Tool, ReadFileTool, WriteFileTool, BashTool')
print('  后续 Lab: from utils.tools import Tool, ReadFileTool, WriteFileTool, BashTool')


In [ ]:
def run_agent_loop(system_prompt, tools_list, user_messages, max_turns=20):
    """Agent Loop（notebook 友好版：消息列表驱动，无 input()）"""
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    elif not tool.validate(blk.input):
                        result, is_error = "参数校验失败", True
                    else:
                        result, is_error = tool.execute(blk.input), False
                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {result[:300]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    print("--- Agent Loop 结束 ---")
    return last_text

print("run_agent_loop 就绪")


---
## 第五步：验证运行

现在让我们测试这个有工具的 agent。试试这些指令：

1. "列出当前目录的文件"
2. "创建一个 hello.py 文件，内容是 print('hello world')"
3. "运行 hello.py"
4. "读取 hello.py 的内容"

你会看到 agent 能**真正执行**这些操作了！

In [ ]:
# 运行带工具的 Agent Loop

SYSTEM_PROMPT = """你是一个编程助手，可以读写文件和执行 bash 命令。
请用中文回答。当需要操作文件或执行命令时，使用提供的工具。"""

print("=" * 60)
print("Lab 2: Agent Loop + 工具执行层")
print("=" * 60)

run_agent_loop(
    system_prompt=SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=[
        "列出当前目录下的文件",
        "创建一个 hello.py 文件，内容是打印当前系统日期和时间",
        "运行 python hello.py",
        "读取 hello.py 的内容",
    ],
)


---
## 完整集成示例：工具 + 项目记忆 + Hooks

到目前为止我们实现了 Lab 1 的提示引导层和 Lab 2 的工具执行层。
下面的 cell 把它们全部集成起来：

- **CLAUDE.md 项目记忆** → 自动注入 system prompt
- **Hooks 日志钩子** → 每次工具调用自动打印日志
- **工具执行层** → 真正执行 read_file / write_file / bash


In [ ]:
# === 完整集成示例：Lab 1 (提示+记忆+Hooks) + Lab 2 (工具) ===
# 直接从 utils/ 导入 Lab 1 的实现，保证代码一致性

import os, json
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner

# ❶ 项目记忆注入 system prompt
pm = ProjectMemory()
full_prompt = pm.build_system_prompt(
    "你是一个编程助手。用中文回答。", os.getcwd())
print(f"System prompt: {len(full_prompt)} 字符（含项目记忆）")

# ❶ Hooks: 日志钩子
hooks = HookRunner()

def log_pre(name, inp):
    print(f"  [Hook-Pre] 即将执行 {name}")
    return inp
def log_post(name, inp, result):
    print(f"  [Hook-Post] {name} 完成，结果长度: {len(result)}")
    return result

hooks.register_pre_tool(log_pre)
hooks.register_post_tool(log_post)

# ❷ 增强版 Agent Loop：集成 Hooks
def run_agent_loop_with_hooks(system_prompt, tools_list, user_messages, hooks_runner=None):
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages, msg_index = [], 0
    for turn in range(1, 20):
        if msg_index >= len(user_messages): break
        user_input = user_messages[msg_index]; msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})
        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages)
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool and tool.validate(blk.input):
                        inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
                        if inp is None:
                            result, is_error = "被 Hook 阻止", True
                        else:
                            result, is_error = tool.execute(inp), False
                            if hooks_runner:
                                result = hooks_runner.run_post_tool(blk.name, inp, result)
                    else:
                        result, is_error = f"工具错误: {blk.name}", True
                    print(f"  [{blk.name}] -> {result[:200]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {})})
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use": break
    print("\n--- 完整集成示例结束 ---")

# 运行
print("=" * 60)
print("完整集成：❶提示引导(CLAUDE.md+Hooks) + ❷工具执行")
print("=" * 60)
run_agent_loop_with_hooks(
    system_prompt=full_prompt,
    tools_list=TOOLS,
    hooks_runner=hooks,
    user_messages=["列出当前目录的文件", "删除hello.py"],
)


---
## 小结

### 本 Lab 你学到了什么

1. **统一工具接口（Factory Pattern）** — 所有工具实现同一个 `Tool` 基类
   - `name` + `description` + `input_schema` + `validate()` + `execute()`
   - 新增工具零改动 Agent Loop

2. **工具执行管道** — tool_use → 查找 → 校验 → 执行 → tool_result → 继续循环

3. **Agent 现在能真正做事了** — 读文件、写文件、执行命令

4. **完整集成** — 工具执行层 + Lab 1 的提示引导（CLAUDE.md + Hooks）协同工作

### 痛点预告

现在试试对 agent 说：「删除当前目录所有 .py 文件」

它会**毫不犹豫地执行** `rm *.py` — 没有任何安全检查！

→ **下一个 Lab：Lab 3 验证与安全层 — 给 agent 装上安全带，拦截危险操作。**
